In [6]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb spacy tqdm
!pip install --upgrade contractions

In [7]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map
import random

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login()  # reads WANDB_API_KEY from the environment  # Replace with your actual API key
wandb.init(project="Spring2025", reinit=True)


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [8]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

Using cuda for training


In [9]:
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [10]:
import spacy

# Load spaCy model with GPU support
spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available

    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/content/279_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)



spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [41:06<00:00,  3.41it/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [11]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [12]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=279).to(device)

# Adjusted LoRA Configuration for Long-Form Text Classification
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.8,
    target_modules=[
        "attention.self.query",
        "attention.self.key",
        "attention.self.value",
        "attention.output.dense",
        "intermediate.dense",
        "output.dense"
    ],
    bias="none",
    inference_mode=False
)

# Integrate LoRA with the model
model = get_peft_model(model, lora_config).to(device)
model.print_trainable_parameters()


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,868,759 || all params: 112,565,550 || trainable%: 2.5485


In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [14]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/results",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="/kaggle/working/logs",
#     label_names=["labels"],
#     logging_steps=10,
#     num_train_epochs=20,  # More epochs for better legal text learning
#     per_device_train_batch_size=4,  # Lower batch size for stability
#     per_device_eval_batch_size=4,
#     learning_rate=2e-5,  # Reduce LR for better fine-tuning
#     warmup_ratio=0.1,  # Stabilize early training
#     weight_decay=0.01,
#     metric_for_best_model="balanced_accuracy",
#     load_best_model_at_end=True,
#     report_to=["wandb"],  # Enable W&B logging
#     fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
# )

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",        # Save the model every epoch
    learning_rate=2e-5,           # Reduced LR for stability
    per_device_train_batch_size=16,  # Increase if memory allows
    per_device_eval_batch_size=16,
    num_train_epochs=20,         # Keep training epochs
    weight_decay=0.05,           # Regularization to prevent overfitting
    logging_dir="./logs",
    logging_steps=50,            # Log progress every 50 steps
    load_best_model_at_end=True, # Load the best checkpoint
    metric_for_best_model="eval_loss",  # Select the best model based on validation loss
    lr_scheduler_type="cosine",  # Cosine decay learning rate
    warmup_ratio=0.1,            # 10% warmup steps
    gradient_accumulation_steps=2,  # Stabilizes training for smaller batch sizes
    fp16=True,                   # Mixed precision training
    report_to="wandb",           # Log results to Weights & Biases
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [15]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,5.564700,5.492067,0.021378,0.006509,0.001950,0.021378,0.003490
2,5.161200,5.103866,0.024941,0.005025,0.000624,0.024941,0.001218
3,5.074100,5.061282,0.030879,0.005764,0.004600,0.030879,0.005727
4,5.084200,5.042486,0.040380,0.008269,0.002968,0.040380,0.005470
5,5.017900,4.965635,0.068884,0.014249,0.012030,0.068884,0.018828
6,4.844300,4.807095,0.081948,0.016782,0.019187,0.081948,0.021843
7,4.690300,4.659349,0.096200,0.020197,0.018351,0.096200,0.027725
8,4.561800,4.527070,0.112827,0.025780,0.023422,0.112827,0.036705
9,4.452500,4.419280,0.125891,0.030238,0.026007,0.125891,0.041383
10,4.303900,4.298960,0.142518,0.033757,0.033513,0.142518,0.048168


/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:1107: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 367e9cae-5be4-4df4-a67a-6cc62e975156)') - silently ignoring the lookup for the file config.json in bert-base-uncased.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in bert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'eval_loss': 4.00558614730835, 'eval_accuracy': 0.171021377672209, 'eval_balanced_accuracy': 0.0458078177525292, 'eval_precision': 0.04230011443262657, 'eval_recall': 0.171021377672209, 'eval_f1': 0.06380131323812381, 'eval_runtime': 10.7823, 'eval_samples_per_second': 78.091, 'eval_steps_per_second': 4.915, 'epoch': 20.0}
